# ALLMAMBA -- Ensemble Notebook
Train **all 12 Mamba models** (9 baseline + 3 novel) and run full ensemble comparison.

Designed to run identically on **Kaggle** (git clone) or **local** (auto-detect repo root).

| Group | Models |
|-------|--------|
| Baseline (9) | Mamba1, Mamba2, Mamba3, VMamba, MambaVision, MedMamba, VSSD, DSA-Mamba, EfficientMamba |
| Novel (3)    | AdaptiveScanMamba, MoEMamba, CrossFusionMamba |

Ensemble techniques: `simple_avg` · `weighted_avg` · `top_k` · `majority_vote` · `soft_vote` · `stacking`


## Section 1 -- CONFIGURATION  (Edit only here)

In [ ]:
# ==============================================================
#         ALL VARIABLES -- EDIT ONLY THIS BLOCK
# ==============================================================

# ── Task ─────────────────────────────────────────────────────
TASK = "classification"   # "classification" | "regression"

# ── Dataset 1 (primary) ──────────────────────────────────────
IMAGE_DIR = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye"
CSV_PATH  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx"

# ── Dataset 2 (optional -- leave empty to disable) ───────────
IMAGE_DIR_2 = ""
CSV_PATH_2  = ""

# ── Column names ─────────────────────────────────────────────
IMAGE_COL = "Patient ID"
HB_COL    = "Haemoglobin (gm/dL)"
HB_THRESH = 12.0

# ── HB Range Filter ──────────────────────────────────────────
HB_FILTER_MIN = 7.0
HB_FILTER_MAX = 15.0

# ── Augmentation ─────────────────────────────────────────────
USE_AUGMENTATION = True
AUG_BINS         = 10

# ── Training ─────────────────────────────────────────────────
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-4
VAL_SPLIT   = 0.2
TEST_SPLIT  = 0.1
NUM_WORKERS = 4
SEED        = 42
EARLY_STOP  = 8
SCHEDULER   = "cosine"

# ── Image size ────────────────────────────────────────────────
IMG_SIZE = 224

# ── Preprocessing ────────────────────────────────────────────
PREPROCESS = {
    "colorspace" : "RGB",   # "RGB" | "CIELAB" | "HSV" | "GRAY"
    "use_clahe"  : False,
    "clahe_clip" : 2.0,
    "clahe_grid" : (8, 8),
}

# ── Classification settings ──────────────────────────────────
CLS_CONFIG = {
    "use_focal_loss"    : True,
    "focal_gamma"       : 2.0,
    "use_class_weights" : True,
    "threshold"         : 0.5,
}

# ── Regression settings ──────────────────────────────────────
REG_CONFIG = {
    "normalize_hb"  : True,
    "hb_min"        : 4.0,
    "hb_max"        : 20.0,
    "loss_fn"       : "huber",
    "huber_delta"   : 1.0,
}

# ── Which models to run ──────────────────────────────────────
RUN = {
    # Baseline models
    "Mamba1"        : True,
    "Mamba2"        : True,
    "Mamba3"        : True,
    "VMamba"        : True,
    "MambaVision"   : True,
    "MedMamba"      : True,
    "VSSD"          : True,
    "DSA_Mamba"     : True,
    "EfficientMamba": True,
    # Novel models
    "AdaptiveScan"  : True,
    "MoEMamba"      : True,
    "CrossFusion"   : True,
}

# ── Novel model config (new architectures) ───────────────────
NEW_MODEL_CFG = {
    "embed_dim"   : 128,
    "depth"       : 4,
    "d_state"     : 64,        # richer SSM state (matches best Mamba2 result)
    "colour_proj" : "learned", # "RGB" | "learned" | "pallor" | "both"
    "n_experts"   : 5,         # 4 = (R,G,B,Lum) | 5 = (R,G,B,Lum,PallorIndex)
}

# ── Training improvements ─────────────────────────────────────
GRAD_ACCUM      = 1
WARMUP_EPOCHS   = 3
USE_AMP         = True
FREEZE_BACKBONE = True    # EfficientMamba only

# ── Ensemble ─────────────────────────────────────────────────
ENSEMBLE_TECHNIQUE = "auto"   # "auto" runs all techniques and picks best
ENSEMBLE_TOP_K     = 3

# ── Paths (auto-detected) ─────────────────────────────────────
import os, sys

def _find_repo_root():
    candidates = [
        os.getcwd(),
        os.path.join(os.getcwd(), "ALLMAMBA"),
    ]
    try:
        for d in os.listdir(os.getcwd()):
            p = os.path.join(os.getcwd(), d)
            if os.path.isdir(p):
                candidates.append(p)
    except Exception:
        pass
    candidates.append("/kaggle/working/ALLMAMBA")
    for c in candidates:
        if os.path.isdir(os.path.join(c, "MAMBA_MODELS")):
            return c
    return os.getcwd()

REPO_ROOT  = _find_repo_root()
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")

if not os.path.isdir(MODELS_DIR):
    raise RuntimeError(
        f"Cannot find MAMBA_MODELS.\n"
        f"  cwd: {os.getcwd()}\n  REPO_ROOT: {REPO_ROOT}\n"
        f"  Expected: {MODELS_DIR}\n"
        "  Run the git-clone cell first.")

if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)

print(f"TASK       : {TASK}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"MODELS_DIR : {MODELS_DIR}")
print(f"Models ON  : {[k for k, v in RUN.items() if v]}")


# ── Cross-Validation ──────────────────────────────────────────────────────────
USE_CROSSVAL = True      # True = 5-fold CV (publication mode)
N_FOLDS      = 5         # Folds (5 is standard for medical imaging papers)

# ── CNN & ViT Baseline Models ─────────────────────────────────────────────────
RUN_BASELINES = {
    "ResNet50"      : True,   # He et al., CVPR 2016  (~25M params)
    "EfficientNetB3": True,   # Tan & Le, ICML 2019   (~12M params)
    "DenseNet121"   : True,   # Huang et al., CVPR 2017 (~8M params) — CheXNet base
    "MobileNetV3"   : True,   # Howard et al., ICCV 2019 (~5M params)
    "ViT_B16"       : True,   # Dosovitskiy et al., ICLR 2021 (~86M params)
}

# ── Ensemble model selection — toggle ON/OFF to build your ensemble pool ──────
# Run the Ensemble cell at the bottom to see all combinations automatically.
ENSEMBLE_RUN = {
    # ── Mamba Models (12) ──────────────────────────────────────────────────
    "Mamba1"        : True,
    "Mamba2"        : True,
    "Mamba3"        : True,
    "VMamba"        : True,
    "MambaVision"   : True,
    "MedMamba"      : True,
    "VSSD"          : True,
    "DSA_Mamba"     : True,
    "EfficientMamba": True,
    "AdaptiveScan"  : True,
    "MoEMamba"      : True,
    "CrossFusion"   : True,
    # ── CNN Baselines (4) ──────────────────────────────────────────────────
    "ResNet50"      : True,
    "EfficientNetB3": True,
    "DenseNet121"   : True,
    "MobileNetV3"   : True,
    # ── Transformer Baseline (1) ───────────────────────────────────────────
    "ViT_B16"       : True,
}

print(f"USE_CROSSVAL : {USE_CROSSVAL}  ({N_FOLDS}-fold)")
print(f"Baselines ON : {[k for k,v in RUN_BASELINES.items() if v]}")
print(f"Ensemble ON  : {[k for k,v in ENSEMBLE_RUN.items() if v]}")


## Step 0 -- Clone Repo  (run once on Kaggle, skip if already cloned)

In [ ]:
import subprocess

REPO_URL  = "https://github.com/junaidmaqbool/ALLMAMBA.git"
CLONE_DIR = "/kaggle/working/ALLMAMBA"

if not os.path.isdir(CLONE_DIR):
    print("Cloning ALLMAMBA ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    REPO_ROOT  = CLONE_DIR
    MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
    if MODELS_DIR not in sys.path:
        sys.path.insert(0, MODELS_DIR)
    print(f"Cloned. REPO_ROOT = {REPO_ROOT}")
else:
    print("Repo already cloned. Good to go.")


## Section 2 -- Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "mamba-ssm",      # compiled CUDA kernels for Mamba1/2 (Kaggle T4/P100)
    "causal-conv1d",  # required by mamba-ssm
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]
for pkg in PKGS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'OK  ' if r.returncode==0 else 'WARN'} {pkg}")
print("Dependencies done.")


## Section 3 -- Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, mean_absolute_error, f1_score)
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task   : {TASK}")

if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)


In [ ]:
import numpy as np

if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)
from common.augment import (
    merge_and_filter_datasets,
    make_balanced_loader,
)


class CLAHETransform:
    def __call__(self, img):
        import cv2
        img_np = np.array(img)
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe  = cv2.createCLAHE(clipLimit=PREPROCESS["clahe_clip"],
                                   tileGridSize=PREPROCESS["clahe_grid"])
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


class ToLABTensor:
    def __call__(self, img):
        import cv2
        img_np = np.array(img).astype(np.uint8)
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)
        lab[:, :, 0]  /= 255.0
        lab[:, :, 1]   = (lab[:, :, 1] - 128.0) / 127.0
        lab[:, :, 2]   = (lab[:, :, 2] - 128.0) / 127.0
        return torch.from_numpy(lab.transpose(2, 0, 1))


class ToHSVTensor:
    def __call__(self, img):
        import cv2
        img_np = np.array(img).astype(np.uint8)
        hsv    = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] /= 179.0
        hsv[:, :, 1] /= 255.0
        hsv[:, :, 2] /= 255.0
        return torch.from_numpy(hsv.transpose(2, 0, 1))


def build_transforms(augment: bool):
    steps = []
    if augment:
        steps += [
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
        ]
    else:
        steps.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))

    if PREPROCESS["use_clahe"]:
        steps.append(CLAHETransform())

    cs = PREPROCESS["colorspace"].upper()
    if cs == "RGB":
        steps += [
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    elif cs == "CIELAB":
        steps.append(ToLABTensor())
    elif cs == "HSV":
        steps.append(ToHSVTensor())
    elif cs in ("GRAY", "GRAYSCALE"):
        steps += [
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    else:
        raise ValueError(f"Unknown colorspace '{PREPROCESS['colorspace']}'.")
    return transforms.Compose(steps)


_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]

def find_image_path(pid: str, img_dir: str):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p):
            return p
    return None


class EyeHBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        hb    = float(row[HB_COL])
        label = 0 if hb < HB_THRESH else 1
        img   = Image.open(row["_img_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return (
            img,
            torch.tensor(label, dtype=torch.long),
            torch.tensor(hb,    dtype=torch.float32),
        )


def load_data():
    slot_dfs, slot_dirs = [], []
    for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
        if csv_p and img_d:
            _df = (pd.read_excel(csv_p)
                   if csv_p.endswith((".xlsx", ".xls"))
                   else pd.read_csv(csv_p))
            slot_dfs.append(_df)
            slot_dirs.append(img_d)

    if not slot_dfs:
        raise RuntimeError("No dataset slots active. Set CSV_PATH and IMAGE_DIR.")

    df = merge_and_filter_datasets(
        dataframes=slot_dfs,
        image_dirs=slot_dirs,
        hb_col=HB_COL,
        image_col=IMAGE_COL,
        hb_filter_min=HB_FILTER_MIN,
        hb_filter_max=HB_FILTER_MAX,
        find_image_path_fn=find_image_path,
        verbose=True,
    )

    if len(df) == 0:
        raise RuntimeError("Zero valid samples after filtering. Check IMAGE_DIR and CSV_PATH.")

    binary_lb = (df[HB_COL] < HB_THRESH).astype(int)
    n_anemic  = int(binary_lb.sum())
    n_normal  = len(df) - n_anemic
    ratio     = n_anemic / len(df)
    print(f"  Anemic  (HB < {HB_THRESH}): {n_anemic:4d}  ({ratio*100:.1f}%)")
    print(f"  Normal  (HB >= {HB_THRESH}): {n_normal:4d}  ({(1-ratio)*100:.1f}%)")
    print(f"  HB -> mean={df[HB_COL].mean():.2f}  "
          f"min={df[HB_COL].min():.2f}  max={df[HB_COL].max():.2f}  "
          f"std={df[HB_COL].std():.2f} g/dL")

    tr_vl_df, ts_df = train_test_split(
        df, test_size=TEST_SPLIT, random_state=SEED, stratify=binary_lb)
    tr_vl_lb = binary_lb.loc[tr_vl_df.index]
    val_ratio = VAL_SPLIT / (1.0 - TEST_SPLIT)
    tr_df, vl_df = train_test_split(
        tr_vl_df, test_size=val_ratio, random_state=SEED, stratify=tr_vl_lb)

    T_TRAIN = build_transforms(augment=True)
    T_VAL   = build_transforms(augment=False)
    print(f"  Split: Train={len(tr_df)}  Val={len(vl_df)}  Test={len(ts_df)}")

    kw_common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=(DEVICE == "cuda"), drop_last=False)
    val_loader  = DataLoader(EyeHBDataset(vl_df, T_VAL), shuffle=False, **kw_common)
    test_loader = DataLoader(EyeHBDataset(ts_df, T_VAL), shuffle=False, **kw_common)

    tr_dataset = EyeHBDataset(tr_df, T_TRAIN)
    if USE_AUGMENTATION:
        train_loader = make_balanced_loader(
            dataset=tr_dataset,
            hb_values=tr_df[HB_COL].values,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            device=DEVICE,
            hb_filter_min=HB_FILTER_MIN,
            hb_filter_max=HB_FILTER_MAX,
            aug_bins=AUG_BINS,
        )
        print(f"  Augmentation: ON  ({AUG_BINS} bins)")
    else:
        train_loader = DataLoader(tr_dataset, shuffle=True, **kw_common)
        print("  Augmentation: OFF")

    global _TRAIN_LABELS
    _TRAIN_LABELS = binary_lb.loc[tr_df.index].tolist()
    return train_loader, val_loader, test_loader


_TRAIN_LABELS                       = []
TRAIN_LOADER, VAL_LOADER, TEST_LOADER = load_data()


In [ ]:
# ── K-Fold data setup ─────────────────────────────────────────────────────────
# Prepares the FIXED held-out test set (10%) and the full train+val dataframe
# (90%) used by run_model_kfold().  Only runs when USE_CROSSVAL=True.
from sklearn.model_selection import StratifiedKFold

KFOLD_FULL_DF   = None
KFOLD_FULL_LBLS = None

def load_data_for_kfold():
    slot_dfs, slot_dirs = [], []
    for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
        if csv_p and img_d:
            _df = (pd.read_excel(csv_p)
                   if csv_p.endswith((".xlsx", ".xls"))
                   else pd.read_csv(csv_p))
            slot_dfs.append(_df)
            slot_dirs.append(img_d)
    if not slot_dfs:
        raise RuntimeError("No dataset slots active.")
    df = merge_and_filter_datasets(
        dataframes=slot_dfs, image_dirs=slot_dirs,
        hb_col=HB_COL, image_col=IMAGE_COL,
        hb_filter_min=HB_FILTER_MIN, hb_filter_max=HB_FILTER_MAX,
        find_image_path_fn=find_image_path, verbose=False,
    )
    binary_lb = (df[HB_COL] < HB_THRESH).astype(int)
    tr_df, ts_df = train_test_split(
        df, test_size=TEST_SPLIT, random_state=SEED, stratify=binary_lb)
    T_VAL = build_transforms(augment=False)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE=="cuda"), drop_last=False)
    test_loader = DataLoader(EyeHBDataset(ts_df, T_VAL), shuffle=False, **kw)
    tr_lbls = (tr_df[HB_COL] < HB_THRESH).astype(int).values
    print(f"  KFold setup -> Train+Val pool: {len(tr_df)}  |  Fixed Test: {len(ts_df)}")
    return tr_df.reset_index(drop=True), tr_lbls, test_loader


if USE_CROSSVAL:
    KFOLD_FULL_DF, KFOLD_FULL_LBLS, TEST_LOADER = load_data_for_kfold()
    print(f"  {N_FOLDS}-Fold CV ready  |  test set locked at {len(TEST_LOADER.dataset)} samples")
else:
    print("  USE_CROSSVAL=False  ->  using standard TRAIN/VAL/TEST split from load_data()")


## Section 4 -- Training Utilities

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def _norm_hb(hb):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return (hb - lo) / (hi - lo)

def _denorm_hb(hb_n):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return hb_n * (hi - lo) + lo


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()


def _build_loss_fns():
    global CLS_FN, REG_FN
    if TASK == "classification":
        wt = None
        if CLS_CONFIG["use_class_weights"] and len(_TRAIN_LABELS) > 0:
            w  = compute_class_weight("balanced",
                                       classes=np.array([0, 1]),
                                       y=np.array(_TRAIN_LABELS))
            wt = torch.tensor(w, dtype=torch.float32).to(DEVICE)
            print(f"  Class weights -> Anemic: {wt[0]:.3f}  Normal: {wt[1]:.3f}")
        if CLS_CONFIG["use_focal_loss"]:
            CLS_FN = FocalLoss(gamma=CLS_CONFIG["focal_gamma"], weight=wt)
            print(f"  Loss: FocalLoss  (gamma={CLS_CONFIG['focal_gamma']})")
        else:
            CLS_FN = nn.CrossEntropyLoss(weight=wt)
            print(f"  Loss: CrossEntropyLoss")
    elif TASK == "regression":
        fn = REG_CONFIG["loss_fn"].lower()
        if fn == "huber":
            REG_FN = nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
        elif fn == "mae":
            REG_FN = nn.L1Loss()
        else:
            REG_FN = nn.MSELoss()
        print(f"  Loss: {fn.upper()}")

CLS_FN = None
REG_FN = None
_build_loss_fns()


def compute_loss(logits, hb_pred, labels, hb_true):
    if TASK == "classification":
        return CLS_FN(logits, labels)
    hp = hb_pred.view(-1)
    ht = hb_true.view(-1)
    if REG_CONFIG["normalize_hb"]:
        ht = _norm_hb(ht)
    return REG_FN(hp, ht)


class EarlyStopper:
    def __init__(self, patience):
        self.patience   = patience
        self.counter    = 0
        self.best_score = float("inf")

    def step(self, score):
        if score < self.best_score - 1e-6:
            self.best_score = score
            self.counter    = 0
            return False
        self.counter += 1
        return (self.patience > 0) and (self.counter >= self.patience)


def run_epoch(model, loader, optimizer=None, grad_accum=1, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds, all_labels, all_scores = [], [], []
    all_hbp, all_hbt = [], []

    if training:
        optimizer.zero_grad()

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch_idx, (imgs, labels, hb_true) in enumerate(
                tqdm(loader, leave=False, desc="train" if training else "val  ")):

            imgs, labels, hb_true = (imgs.to(DEVICE),
                                     labels.to(DEVICE),
                                     hb_true.to(DEVICE))
            import contextlib
            _amp_ctx = torch.cuda.amp.autocast() if scaler is not None else contextlib.nullcontext()
            with _amp_ctx:
                logits, hb_pred = model(imgs)
                loss = compute_loss(logits, hb_pred, labels, hb_true)

            if training:
                if scaler is not None:
                    scaler.scale(loss / grad_accum).backward()
                else:
                    (loss / grad_accum).backward()

                is_last_batch = (batch_idx + 1 == len(loader))
                if (batch_idx + 1) % grad_accum == 0 or is_last_batch:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    optimizer.zero_grad()

            total_loss += loss.item()

            if TASK == "classification":
                probs = torch.softmax(logits, dim=1)
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
                all_scores.extend(probs[:, 1].cpu().tolist())
            elif TASK == "regression":
                hp = hb_pred.detach().view(-1)
                if REG_CONFIG["normalize_hb"]:
                    hp = _denorm_hb(hp)
                all_hbp.extend(hp.cpu().tolist())
                all_hbt.extend(hb_true.cpu().view(-1).tolist())

    metrics = {"loss": total_loss / max(len(loader), 1)}
    if TASK == "classification":
        metrics["acc"] = accuracy_score(all_labels, all_preds)
        metrics["f1"]  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        try:    metrics["auc"] = roc_auc_score(all_labels, all_scores)
        except: metrics["auc"] = float("nan")
    elif TASK == "regression":
        at, ap = np.array(all_hbt), np.array(all_hbp)
        metrics["mae"]  = float(np.mean(np.abs(at - ap)))
        metrics["rmse"] = float(np.sqrt(np.mean((at - ap) ** 2)))

    return metrics, all_labels, all_preds, all_hbt, all_hbp, all_scores


def find_best_threshold(labels, scores):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.01):
        preds = [1 if s >= t else 0 for s in scores]
        f = f1_score(labels, preds, average="macro", zero_division=0)
        if f > best_f:
            best_f, best_t = f, round(float(t), 2)
    return best_t, best_f


RESULTS          = []
TEST_PREDICTIONS = {}
TRAINED_MODELS   = {}

def run_model(name, model):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    sep    = "=" * 65
    print(f"\n{sep}\n  {name}\n  Trainable: {params/1e6:.2f}M  |  Task: {TASK}")
    print(f"  Grad accum: {GRAD_ACCUM}x  Warmup: {WARMUP_EPOCHS}ep  AMP: {USE_AMP}")
    print(sep)

    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    if WARMUP_EPOCHS > 0 and EPOCHS > WARMUP_EPOCHS:
        from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
        warmup = LinearLR(opt, start_factor=0.05, end_factor=1.0,
                          total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP_EPOCHS, 1))
        sch    = SequentialLR(opt, schedulers=[warmup, cosine],
                               milestones=[WARMUP_EPOCHS])
    elif SCHEDULER == "plateau":
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    elif SCHEDULER == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    else:
        sch = None

    scaler  = (torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None)
    stopper = EarlyStopper(patience=EARLY_STOP)
    hist    = {k: [] for k in ["tr_loss","vl_loss","acc","auc","f1","mae","rmse","lr"]}
    t0      = time.time()
    vl_y = vl_p = vl_scores = None

    for ep in range(1, EPOCHS + 1):
        tr, *_, _        = run_epoch(model, TRAIN_LOADER, opt, grad_accum=GRAD_ACCUM, scaler=scaler)
        vl, vl_y, vl_p, vl_hbt, vl_hbp, vl_scores = run_epoch(model, VAL_LOADER)

        cur_lr = opt.param_groups[0]["lr"]
        if sch is not None:
            if SCHEDULER == "plateau":
                sch.step(vl["loss"])
            else:
                sch.step()

        hist["tr_loss"].append(tr["loss"])
        hist["vl_loss"].append(vl["loss"])
        hist["lr"].append(cur_lr)
        for k in ["acc","auc","f1","mae","rmse"]:
            hist[k].append(vl.get(k, float("nan")))

        line = (f"  Ep {ep:02d}/{EPOCHS}  "
                f"TL:{tr['loss']:.4f}  VL:{vl['loss']:.4f}  LR:{cur_lr:.2e}")
        if TASK == "classification":
            line += (f"  Acc:{vl.get('acc',0):.3f}"
                     f"  AUC:{vl.get('auc',0):.3f}"
                     f"  F1:{vl.get('f1',0):.3f}")
        elif TASK == "regression":
            line += f"  MAE:{vl.get('mae',0):.3f}  RMSE:{vl.get('rmse',0):.3f}"
        print(line)

        if stopper.step(vl["loss"]):
            print(f"  Early stop at epoch {ep}.")
            break

    elapsed = time.time() - t0

    if TASK == "classification" and vl_y:
        print("\n" + "─"*60)
        print("  VAL SET -- Classification Report  (threshold=0.5)")
        print("─"*60)
        print(classification_report(vl_y, vl_p,
                                     target_names=["Anemic","Normal"], zero_division=0))
        if vl_scores:
            best_t, best_f1 = find_best_threshold(vl_y, vl_scores)
            print(f"  Optimal threshold: {best_t:.2f}  macro-F1={best_f1:.3f}")
        else:
            best_t, best_f1 = 0.5, vl.get("f1", 0)
    else:
        best_t, best_f1 = float("nan"), float("nan")

    ts, ts_y, ts_p, ts_hbt, ts_hbp, ts_scores = run_epoch(model, TEST_LOADER)
    sep2 = "═" * 60
    print(f"\n{sep2}\n  TEST SET  --  {name}\n{sep2}")
    if TASK == "classification":
        print(f"  Accuracy : {ts.get('acc', float('nan')):.4f}")
        print(f"  AUC-ROC  : {ts.get('auc', float('nan')):.4f}")
        print(f"  Macro-F1 : {ts.get('f1',  float('nan')):.4f}")
        print()
        print(classification_report(ts_y, ts_p,
                                     target_names=["Anemic","Normal"], zero_division=0))
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(ts_y, ts_p)
        fig_cm, ax_cm = plt.subplots(figsize=(4, 3))
        im = ax_cm.imshow(cm, cmap="Blues")
        ax_cm.set_xticks([0,1]); ax_cm.set_yticks([0,1])
        ax_cm.set_xticklabels(["Anemic","Normal"])
        ax_cm.set_yticklabels(["Anemic","Normal"])
        ax_cm.set_xlabel("Predicted"); ax_cm.set_ylabel("Actual")
        ax_cm.set_title(f"CM -- {name.split()[0]} (Test)")
        for i in range(2):
            for j in range(2):
                ax_cm.text(j, i, str(cm[i,j]), ha="center", va="center",
                           color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=13)
        plt.colorbar(im, ax=ax_cm); plt.tight_layout()
        cm_fname = f"cm_{name.split()[0].replace('/','_')}.png"
        plt.savefig(cm_fname, dpi=100, bbox_inches="tight"); plt.show()
    elif TASK == "regression":
        print(f"  MAE  : {ts.get('mae',  float('nan')):.4f} g/dL")
        print(f"  RMSE : {ts.get('rmse', float('nan')):.4f} g/dL")
    print(sep2)

    TEST_PREDICTIONS[name] = {
        "labels": ts_y,   "preds":  ts_p,   "scores": ts_scores,
        "hbp":    ts_hbp, "hbt":    ts_hbt,
    }
    TRAINED_MODELS[name] = model.cpu()

    last = {k: (v[-1] if v else float("nan")) for k, v in hist.items()}
    RESULTS.append(dict(
        name=name, status="PASS", params=params, time_s=elapsed,
        best_threshold=best_t, best_f1_thresh=best_f1,
        test_acc=ts.get("acc", float("nan")),
        test_auc=ts.get("auc", float("nan")),
        test_f1=ts.get("f1",   float("nan")),
        test_mae=ts.get("mae",  float("nan")),
        test_rmse=ts.get("rmse",float("nan")),
        **{k: v for k, v in last.items() if k != "lr"},
        error=""
    ))

    ep_x  = list(range(1, len(hist["tr_loss"]) + 1))
    fig, axes = plt.subplots(1, 3, figsize=(16, 3))
    axes[0].plot(ep_x, hist["tr_loss"], label="Train")
    axes[0].plot(ep_x, hist["vl_loss"], label="Val")
    axes[0].set_title(f"{name} -- Loss"); axes[0].legend()
    if TASK == "classification":
        axes[1].plot(ep_x, hist["acc"], label="Acc",  color="green")
        axes[1].plot(ep_x, hist["auc"], label="AUC",  color="purple", linestyle="--")
        axes[1].plot(ep_x, hist["f1"],  label="F1",   color="teal",   linestyle=":")
        axes[1].set_title("Metrics"); axes[1].legend(); axes[1].set_ylim(0, 1)
    elif TASK == "regression":
        axes[1].plot(ep_x, hist["mae"],  label="MAE",  color="orange")
        axes[1].plot(ep_x, hist["rmse"], label="RMSE", color="red", linestyle="--")
        axes[1].set_title("Metrics g/dL"); axes[1].legend()
    axes[2].plot(ep_x, hist["lr"], color="steelblue")
    axes[2].set_title("LR Schedule"); axes[2].set_xlabel("Epoch")
    axes[2].ticklabel_format(style="sci", axis="y", scilimits=(0,0))
    plt.suptitle(name, fontsize=11, y=1.02); plt.tight_layout()
    fname = f"result_{name.split()[0].replace('/','_')}.png"
    plt.savefig(fname, dpi=100, bbox_inches="tight"); plt.show()
    print(f"  Done in {elapsed:.0f}s  |  chart -> {fname}")
    return model


def _fail(name, e):
    print(f"  {name} FAILED: {e}")
    traceback.print_exc()
    TEST_PREDICTIONS[name] = None
    RESULTS.append(dict(name=name, status="FAIL", error=str(e),
                        params=0, time_s=0, best_threshold=float("nan"),
                        best_f1_thresh=float("nan"), tr_loss=float("nan"),
                        vl_loss=float("nan"), acc=float("nan"), auc=float("nan"),
                        f1=float("nan"), mae=float("nan"), rmse=float("nan"),
                        test_acc=float("nan"), test_auc=float("nan"),
                        test_f1=float("nan"), test_mae=float("nan"),
                        test_rmse=float("nan")))


def _add(folder: str):
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path:
        sys.path.insert(0, p)


print(f"Training utilities ready.  DEVICE={DEVICE}  GRAD_ACCUM={GRAD_ACCUM}  AMP={USE_AMP}")


# =============================================================================
#  K-FOLD CROSS-VALIDATION  (publication-grade: TEST SET mean+/-std)
#
#  Each fold trains a fresh model on K-1 folds.
#  That fold-model is then evaluated on the FIXED held-out TEST SET.
#  => Mean +/- std of TEST metrics across K folds = what you report in the paper.
#  => Averaged test predictions across K fold-models = final ensemble contribution.
# =============================================================================

def _build_fold_loaders(tr_df, vl_df):
    T_TRAIN = build_transforms(augment=True)
    T_VAL   = build_transforms(augment=False)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE=="cuda"), drop_last=False)
    tr_ds = EyeHBDataset(tr_df.reset_index(drop=True), T_TRAIN)
    vl_loader = DataLoader(EyeHBDataset(vl_df.reset_index(drop=True), T_VAL),
                           shuffle=False, **kw)
    if USE_AUGMENTATION:
        tr_loader = make_balanced_loader(
            dataset=tr_ds, hb_values=tr_df[HB_COL].values,
            batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, device=DEVICE,
            hb_filter_min=HB_FILTER_MIN, hb_filter_max=HB_FILTER_MAX, aug_bins=AUG_BINS)
    else:
        tr_loader = DataLoader(tr_ds, shuffle=True, **kw)
    return tr_loader, vl_loader


def _train_one_fold(model, tr_loader, vl_loader, fold_idx):
    """Train one fold.  Returns (best_val_loss_epoch_metrics, trained_model)."""
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    if WARMUP_EPOCHS > 0 and EPOCHS > WARMUP_EPOCHS:
        from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
        warmup = LinearLR(opt, start_factor=0.05, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP_EPOCHS, 1))
        sch    = SequentialLR(opt, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
    elif SCHEDULER == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    else:
        sch = None
    scaler  = (torch.cuda.amp.GradScaler() if USE_AMP and DEVICE=="cuda" else None)
    stopper = EarlyStopper(patience=EARLY_STOP)
    best_vl_loss = float("inf")
    for ep in range(1, EPOCHS + 1):
        tr, *_ = run_epoch(model, tr_loader, opt, grad_accum=GRAD_ACCUM, scaler=scaler)
        vl, vl_y, vl_p, vl_hbt, vl_hbp, vl_sc = run_epoch(model, vl_loader)
        if sch: sch.step(vl["loss"]) if SCHEDULER=="plateau" else sch.step()
        line = (f"    Fold{fold_idx+1} Ep{ep:02d}/{EPOCHS}"
                f"  TrL:{tr['loss']:.4f}  VlL:{vl['loss']:.4f}")
        if TASK=="classification":
            line += (f"  Acc:{vl.get('acc',0):.3f}"
                     f"  AUC:{vl.get('auc',0):.3f}"
                     f"  F1:{vl.get('f1',0):.3f}")
        else:
            line += f"  MAE:{vl.get('mae',0):.3f}"
        print(line)
        if stopper.step(vl["loss"]):
            print(f"    Early stop fold {fold_idx+1} at epoch {ep}.")
            break
    return model


def run_model_kfold(name, build_fn, *pos_args, **build_kwargs):
    """
    5-Fold CV training.  Reports TEST SET mean+/-std (publication standard).
    Falls back to run_model() when USE_CROSSVAL=False.
    Signature: run_model_kfold(name, build_fn, *pos_args_for_build_fn, **kwargs)
    """
    if not USE_CROSSVAL or KFOLD_FULL_DF is None:
        model = build_fn(*pos_args, **build_kwargs)
        return run_model(name, model)

    global _TRAIN_LABELS, CLS_FN, REG_FN
    sep = "=" * 70
    print(f"\n{sep}\n  {name}  [{N_FOLDS}-Fold CV]\n{sep}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    # Per-fold TEST metrics (for mean+/-std reporting)
    fold_test_accs   = []
    fold_test_aucs   = []
    fold_test_f1s    = []
    fold_test_maes   = []
    fold_test_rmses  = []

    # Predictions for ensemble
    fold_test_scores = []
    fold_test_preds  = []
    fold_test_hbp    = []
    ts_y = ts_hbt = None
    t0 = time.time()

    for fold_idx, (tr_idx, vl_idx) in enumerate(
            skf.split(KFOLD_FULL_DF, KFOLD_FULL_LBLS)):
        print(f"\n  -- Fold {fold_idx+1}/{N_FOLDS} " + "-"*40)
        tr_df = KFOLD_FULL_DF.iloc[tr_idx]
        vl_df = KFOLD_FULL_DF.iloc[vl_idx]

        _TRAIN_LABELS = (tr_df[HB_COL] < HB_THRESH).astype(int).tolist()
        _build_loss_fns()

        tr_loader, vl_loader = _build_fold_loaders(tr_df, vl_df)
        model = build_fn(*pos_args, **build_kwargs)
        trained = _train_one_fold(model, tr_loader, vl_loader, fold_idx)

        # ── Evaluate on FIXED TEST SET ────────────────────────────────────────
        ts, ts_y_, ts_p_, ts_hbt_, ts_hbp_, ts_sc_ = run_epoch(trained, TEST_LOADER)
        if ts_y is None: ts_y, ts_hbt = ts_y_, ts_hbt_

        fold_test_scores.append(ts_sc_)
        fold_test_preds.append(ts_p_)
        fold_test_hbp.append(ts_hbp_)

        if TASK == "classification":
            fold_test_accs.append(ts.get("acc", float("nan")))
            fold_test_aucs.append(ts.get("auc", float("nan")))
            fold_test_f1s.append(ts.get("f1",  float("nan")))
            print(f"    Fold{fold_idx+1} TEST:  Acc={ts.get('acc',0):.4f}"
                  f"  AUC={ts.get('auc',0):.4f}  F1={ts.get('f1',0):.4f}")
        else:
            fold_test_maes.append(ts.get("mae",   float("nan")))
            fold_test_rmses.append(ts.get("rmse",  float("nan")))
            print(f"    Fold{fold_idx+1} TEST:  MAE={ts.get('mae',0):.4f}"
                  f"  RMSE={ts.get('rmse',0):.4f}")

        del trained
        if DEVICE == "cuda": torch.cuda.empty_cache()

    elapsed = time.time() - t0

    # ── Summary: TEST mean +/- std  (this is what you publish) ───────────────
    print(f"\n{'─'*70}")
    print(f"  {name}")
    print(f"  {N_FOLDS}-FOLD CROSS-VALIDATION  --  TEST SET RESULTS  (publication table)")
    print(f"{'─'*70}")

    if TASK == "classification":
        mean_acc,std_acc = float(np.nanmean(fold_test_accs)),  float(np.nanstd(fold_test_accs))
        mean_auc,std_auc = float(np.nanmean(fold_test_aucs)),  float(np.nanstd(fold_test_aucs))
        mean_f1, std_f1  = float(np.nanmean(fold_test_f1s)),   float(np.nanstd(fold_test_f1s))
        print(f"  Per-fold Test Acc : {[f'{v:.4f}' for v in fold_test_accs]}")
        print(f"  Per-fold Test AUC : {[f'{v:.4f}' for v in fold_test_aucs]}")
        print(f"  Per-fold Test F1  : {[f'{v:.4f}' for v in fold_test_f1s]}")
        print(f"\n  Acc = {mean_acc:.4f} +/- {std_acc:.4f}")
        print(f"  AUC = {mean_auc:.4f} +/- {std_auc:.4f}   <-- REPORT THIS IN PAPER")
        print(f"  F1  = {mean_f1:.4f} +/- {std_f1:.4f}")
    else:
        mean_mae,std_mae   = float(np.nanmean(fold_test_maes)),  float(np.nanstd(fold_test_maes))
        mean_rmse,std_rmse = float(np.nanmean(fold_test_rmses)), float(np.nanstd(fold_test_rmses))
        print(f"  Per-fold Test MAE  : {[f'{v:.4f}' for v in fold_test_maes]}")
        print(f"  Per-fold Test RMSE : {[f'{v:.4f}' for v in fold_test_rmses]}")
        print(f"\n  MAE  = {mean_mae:.4f} +/- {std_mae:.4f}   <-- REPORT THIS IN PAPER")
        print(f"  RMSE = {mean_rmse:.4f} +/- {std_rmse:.4f}")

    # ── Ensemble contribution: average of K fold-model predictions ────────────
    avg_scores = np.mean(fold_test_scores, axis=0).tolist() if fold_test_scores else []
    avg_preds  = [1 if s >= 0.5 else 0 for s in avg_scores]
    avg_hbp    = np.mean(fold_test_hbp, axis=0).tolist() if fold_test_hbp else []

    if TASK == "classification" and ts_y:
        from sklearn.metrics import accuracy_score as _a, roc_auc_score as _r, f1_score as _f
        ens_acc = _a(ts_y, avg_preds)
        try:   ens_auc = _r(ts_y, avg_scores)
        except: ens_auc = float("nan")
        ens_f1 = _f(ts_y, avg_preds, average="macro", zero_division=0)
        print(f"\n  Avg-of-{N_FOLDS}-folds ensemble on test:")
        print(f"  Acc={ens_acc:.4f}  AUC={ens_auc:.4f}  F1={ens_f1:.4f}")
        print(classification_report(ts_y, avg_preds,
                                     target_names=["Anemic","Normal"], zero_division=0))
        from sklearn.metrics import confusion_matrix as _cm
        cm_d = _cm(ts_y, avg_preds)
        fig_c, ax_c = plt.subplots(figsize=(4,3))
        im = ax_c.imshow(cm_d, cmap="Blues")
        ax_c.set_xticks([0,1]); ax_c.set_yticks([0,1])
        ax_c.set_xticklabels(["Anemic","Normal"]); ax_c.set_yticklabels(["Anemic","Normal"])
        ax_c.set_xlabel("Predicted"); ax_c.set_ylabel("Actual")
        ax_c.set_title(f"CM {name.split()[0]} (avg {N_FOLDS}-fold, Test)")
        for i in range(2):
            for j in range(2):
                ax_c.text(j,i,str(cm_d[i,j]),ha="center",va="center",
                          color="white" if cm_d[i,j]>cm_d.max()/2 else "black",fontsize=13)
        plt.colorbar(im,ax=ax_c); plt.tight_layout()
        plt.savefig(f"cm_kfold_{name.split()[0].replace('/','_')}.png",
                    dpi=100,bbox_inches="tight"); plt.show()
    elif TASK == "regression" and ts_hbt:
        at=np.array(ts_hbt); ap=np.array(avg_hbp)
        ens_mae  = float(np.mean(np.abs(at-ap)))
        ens_rmse = float(np.sqrt(np.mean((at-ap)**2)))
        print(f"\n  Avg-of-{N_FOLDS}-folds ensemble on test:"
              f"  MAE={ens_mae:.4f}  RMSE={ens_rmse:.4f}")

    # ── Store for ensemble section ────────────────────────────────────────────
    TEST_PREDICTIONS[name] = {
        "labels": ts_y or [], "preds": avg_preds, "scores": avg_scores,
        "hbp": avg_hbp, "hbt": ts_hbt or [],
    }

    params = 0
    try:
        _tmp = build_fn(*pos_args, **build_kwargs)
        params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
        del _tmp
    except Exception: pass

    # RESULTS uses TEST mean+/-std as the primary metrics
    if TASK == "classification":
        RESULTS.append(dict(
            name=name, status="PASS", params=params, time_s=elapsed,
            cv_folds=N_FOLDS,
            test_acc_mean=mean_acc, test_acc_std=std_acc,
            test_auc_mean=mean_auc, test_auc_std=std_auc,
            test_f1_mean=mean_f1,  test_f1_std=std_f1,
            # Legacy fields (for ensemble compatibility)
            best_threshold=0.5, best_f1_thresh=mean_f1,
            test_acc=mean_acc, test_auc=mean_auc, test_f1=mean_f1,
            test_mae=float("nan"), test_rmse=float("nan"),
            tr_loss=float("nan"), vl_loss=float("nan"),
            acc=mean_acc, auc=mean_auc, f1=mean_f1,
            mae=float("nan"), rmse=float("nan"), error=""))
    else:
        RESULTS.append(dict(
            name=name, status="PASS", params=params, time_s=elapsed,
            cv_folds=N_FOLDS,
            test_mae_mean=mean_mae, test_mae_std=std_mae,
            test_rmse_mean=mean_rmse, test_rmse_std=std_rmse,
            best_threshold=float("nan"), best_f1_thresh=float("nan"),
            test_acc=float("nan"), test_auc=float("nan"), test_f1=float("nan"),
            test_mae=mean_mae, test_rmse=mean_rmse,
            tr_loss=float("nan"), vl_loss=float("nan"),
            acc=float("nan"), auc=float("nan"), f1=float("nan"),
            mae=mean_mae, rmse=mean_rmse, error=""))

    print(f"{'─'*70}  Done in {elapsed:.0f}s")
    return None


print("K-Fold CV utilities ready (TEST-SET mean+/-std mode).")


---
## Model 1 -- Mamba1 (Official SSM, Gu & Dao 2023)

In [ ]:
if RUN["Mamba1"]:
    _add("01_Mamba_Official")
    try:
        import importlib.util, os as _os
        _spec = importlib.util.spec_from_file_location(
            "_m1_eye", _os.path.join(MODELS_DIR, "01_Mamba_Official", "eye_hb_model.py"))
        _m1_eye = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_m1_eye)
        run_model_kfold("Mamba1 (official SSM)", _m1_eye.build_mamba1, IMG_SIZE, embed_dim=128, depth=4)
    except Exception as e:
        _fail("Mamba1", e)
else:
    print("Mamba1 skipped.")


---
## Model 2 -- Mamba2 / SSD (Dao & Gu, ICML 2024)

In [ ]:
if RUN["Mamba2"]:
    _add("01_Mamba_Official")
    try:
        import importlib.util, os as _os
        _spec = importlib.util.spec_from_file_location(
            "_m2_eye", _os.path.join(MODELS_DIR, "01_Mamba_Official", "eye_hb_model.py"))
        _m2_eye = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_m2_eye)
        run_model_kfold("Mamba2 (SSD, ICML 2024)", _m2_eye.build_mamba2, IMG_SIZE, embed_dim=128, depth=4)
    except Exception as e:
        _fail("Mamba2", e)
else:
    print("Mamba2 skipped.")


---
## Model 3 -- Mamba3 (MIMO + Rotary SSM, ICLR 2026)

In [ ]:
if RUN["Mamba3"]:
    _add("06_Mamba3_Minimal")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_m3_eye", os.path.join(MODELS_DIR, "06_Mamba3_Minimal", "eye_hb_model.py"))
        m3_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(m3_eye)
        run_model_kfold("Mamba3 (ICLR 2026 Oral)", m3_eye.build_model, IMG_SIZE, embed_dim=128, depth=4)
    except Exception as e:
        _fail("Mamba3", e)
else:
    print("Mamba3 skipped.")


---
## Model 4 -- VMamba (2D Cross-Scan SSM, ICLR 2025)

In [ ]:
if RUN["VMamba"]:
    _add("02_VMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vm_eye", os.path.join(MODELS_DIR, "02_VMamba", "eye_hb_model.py"))
        vm_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vm_eye)
        run_model_kfold("VMamba (2D cross-scan, ICLR 2025)", vm_eye.build_model, IMG_SIZE)
    except Exception as e:
        _fail("VMamba", e)
else:
    print("VMamba skipped.")


---
## Model 5 -- MambaVision (NVIDIA, CVPR 2025)

In [ ]:
if RUN["MambaVision"]:
    _add("03_MambaVision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_mv_eye", os.path.join(MODELS_DIR, "03_MambaVision", "eye_hb_model.py"))
        mv_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mv_eye)
        run_model_kfold("MambaVision (NVIDIA, CVPR 2025)", mv_eye.build_model, IMG_SIZE)
    except Exception as e:
        _fail("MambaVision", e)
else:
    print("MambaVision skipped.")


---
## Model 6 -- MedMamba (Medical Vision Mamba, 2024)

In [ ]:
if RUN["MedMamba"]:
    _add("04_MedMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_med_eye", os.path.join(MODELS_DIR, "04_MedMamba", "eye_hb_model.py"))
        med_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(med_eye)
        run_model_kfold("MedMamba (medical imaging, 2024)", med_eye.build_model, IMG_SIZE)
    except Exception as e:
        _fail("MedMamba", e)
else:
    print("MedMamba skipped.")


---
## Model 7 -- VSSD / VMamba2 (Mamba2 Vision, ICCV 2025)

In [ ]:
if RUN["VSSD"]:
    _add("05_VSSD_Mamba2Vision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vssd_eye", os.path.join(MODELS_DIR, "05_VSSD_Mamba2Vision", "eye_hb_model.py"))
        vssd_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vssd_eye)
        run_model_kfold("VSSD (Mamba2 Vision, ICCV 2025)", vssd_eye.build_model, IMG_SIZE)
    except Exception as e:
        _fail("VSSD", e)
else:
    print("VSSD skipped.")


---
## Model 8 -- DSA-Mamba Custom (dual-task eye/HB, 2024)

In [ ]:
if RUN["DSA_Mamba"]:
    _add("07_DSA_Mamba_Custom")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_dsa_eye", os.path.join(MODELS_DIR, "07_DSA_Mamba_Custom", "eye_hb_model.py"))
        dsa_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(dsa_eye)
        run_model_kfold("DSA-Mamba Custom (eye HB dual-task, 2024)", dsa_eye.build_model, IMG_SIZE)
    except Exception as e:
        _fail("DSA_Mamba", e)
else:
    print("DSA-Mamba skipped.")


---
## Model 9 -- EfficientMamba (Pretrained CNN + Mamba SSM)

In [ ]:
if RUN.get("EfficientMamba", True):
    _add("08_EfficientMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_em_eye", os.path.join(MODELS_DIR, "08_EfficientMamba", "eye_hb_model.py"))
        em_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(em_eye)
        run_model_kfold("EfficientMamba (pretrained B0 + Mamba SSM)",
                  em_eye.build_model,
                  img_size=IMG_SIZE, embed_dim=256, depth=4,
                  freeze_backbone=FREEZE_BACKBONE)
    except Exception as e:
        _fail("EfficientMamba", e)
else:
    print("EfficientMamba skipped.")


---
## Model 10 -- AdaptiveScanMamba  *(Novel)*
Learnable scan-direction routing: each image patch votes which of
4 scan directions (H / V / R-H / R-V) is most informative.
Colour projection modes: `RGB` · `learned` · `pallor` · `both`.


In [ ]:
if RUN.get("AdaptiveScan", True):
    _add("09_AdaptiveScanMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_asm_eye",
            os.path.join(MODELS_DIR, "09_AdaptiveScanMamba", "adaptive_scan_mamba.py"))
        asm_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(asm_mod)
        asm_model = asm_mod.build_adaptive_scan_mamba(
            img_size    = IMG_SIZE,
            embed_dim   = NEW_MODEL_CFG["embed_dim"],
            depth       = NEW_MODEL_CFG["depth"],
            d_state     = NEW_MODEL_CFG["d_state"],
            colour_proj = NEW_MODEL_CFG["colour_proj"],
        )
        run_model_kfold("AdaptiveScanMamba (novel, pallor-aware scan)",
            asm_mod.build_adaptive_scan_mamba,
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"],
            colour_proj=NEW_MODEL_CFG["colour_proj"])
    except Exception as e:
        _fail("AdaptiveScan", e)
else:
    print("AdaptiveScanMamba skipped.")


---
## Model 11 -- MoEMamba  *(Novel)*
Mixture-of-Experts with per-channel SSM experts:
R · G · B · Luminance · PallorIndex=(R-G)/(R+G).
Top-2 gating routes each patch to its 2 most relevant experts.


In [ ]:
if RUN.get("MoEMamba", True):
    _add("10_MoEMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_moe_eye",
            os.path.join(MODELS_DIR, "10_MoEMamba", "moe_mamba.py"))
        moe_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(moe_mod)
        moe_model = moe_mod.build_moe_mamba(
            img_size  = IMG_SIZE,
            embed_dim = NEW_MODEL_CFG["embed_dim"],
            depth     = NEW_MODEL_CFG["depth"],
            n_experts = NEW_MODEL_CFG["n_experts"],
            d_state   = NEW_MODEL_CFG["d_state"],
        )
        run_model_kfold("MoEMamba (novel, channel-expert routing)",
            moe_mod.build_moe_mamba,
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], n_experts=NEW_MODEL_CFG["n_experts"],
            d_state=NEW_MODEL_CFG["d_state"])
    except Exception as e:
        _fail("MoEMamba", e)
else:
    print("MoEMamba skipped.")


---
## Model 12 -- CrossFusionMamba  *(Novel)*
Dual-branch architecture: Branch 1 processes RGB tokens,
Branch 2 processes a pallor-normalised view (colour proportions).
Bidirectional cross-attention fuses clinical signal from both branches.


In [ ]:
if RUN.get("CrossFusion", True):
    _add("11_CrossFusionMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_cf_eye",
            os.path.join(MODELS_DIR, "11_CrossFusionMamba", "cross_fusion_mamba.py"))
        cf_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(cf_mod)
        cf_model = cf_mod.build_cross_fusion_mamba(
            img_size  = IMG_SIZE,
            embed_dim = NEW_MODEL_CFG["embed_dim"],
            depth     = NEW_MODEL_CFG["depth"],
            d_state   = NEW_MODEL_CFG["d_state"],
        )
        run_model_kfold("CrossFusionMamba (novel, dual-branch pallor fusion)",
            cf_mod.build_cross_fusion_mamba,
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"])
    except Exception as e:
        _fail("CrossFusion", e)
else:
    print("CrossFusionMamba skipped.")


---
## Section 5 -- CNN & ViT Baselines (Standard Comparison Models)
Pretrained on ImageNet. Same training pipeline, same loss, same K-fold CV as Mamba models.
TimmWrapper gives identical  output API to all Mamba models.


In [ ]:
# ── TimmWrapper: shared backbone for CNN/ViT baselines ───────────────────────
# Identical output API to all Mamba models: forward(x) -> (logits, hb_pred)
import timm

class TimmWrapper(torch.nn.Module):
    def __init__(self, backbone_name, pretrained=True, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features
        self.cls_head = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim, num_classes))
        self.reg_head = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim, 1))

    def forward(self, x):
        feats   = self.backbone(x)
        logits  = self.cls_head(feats)
        hb_pred = self.reg_head(feats).squeeze(-1)
        return logits, hb_pred


def build_timm_model(backbone_name, pretrained=True):
    return TimmWrapper(backbone_name, pretrained=pretrained)

print(f"TimmWrapper ready.  timm={timm.__version__}")


---
### CNN Baseline 1 -- ResNet-50  (He et al., CVPR 2016)  |  ~25M params
Foundational residual network. Most cited CNN in computer vision history.

In [ ]:
if RUN_BASELINES.get("ResNet50", True):
    try:
        run_model_kfold(
            "ResNet50 (pretrained, CVPR 2016)",
            build_fn=build_timm_model,
            backbone_name="resnet50", pretrained=True)
    except Exception as e:
        _fail("ResNet50", e)
else:
    print("ResNet50 skipped.")


---
### CNN Baseline 2 -- EfficientNet-B3  (Tan & Le, ICML 2019)  |  ~12M params
Compound-scaled CNN. Strong accuracy/efficiency trade-off.

In [ ]:
if RUN_BASELINES.get("EfficientNetB3", True):
    try:
        run_model_kfold(
            "EfficientNetB3 (pretrained, ICML 2019)",
            build_fn=build_timm_model,
            backbone_name="efficientnet_b3", pretrained=True)
    except Exception as e:
        _fail("EfficientNetB3", e)
else:
    print("EfficientNetB3 skipped.")


---
### CNN Baseline 3 -- DenseNet-121  (Huang et al., CVPR 2017)  |  ~8M params
Dense connectivity. Standard medical imaging baseline (CheXNet uses this).

In [ ]:
if RUN_BASELINES.get("DenseNet121", True):
    try:
        run_model_kfold(
            "DenseNet121 (pretrained, CVPR 2017)",
            build_fn=build_timm_model,
            backbone_name="densenet121", pretrained=True)
    except Exception as e:
        _fail("DenseNet121", e)
else:
    print("DenseNet121 skipped.")


---
### CNN Baseline 4 -- MobileNetV3-Large  (Howard et al., ICCV 2019)  |  ~5M params
Lightweight CNN. Shows Mamba models beat even efficiency-optimized CNNs.

In [ ]:
if RUN_BASELINES.get("MobileNetV3", True):
    try:
        run_model_kfold(
            "MobileNetV3 (pretrained, ICCV 2019)",
            build_fn=build_timm_model,
            backbone_name="mobilenetv3_large_100", pretrained=True)
    except Exception as e:
        _fail("MobileNetV3", e)
else:
    print("MobileNetV3 skipped.")


---
### Transformer Baseline -- ViT-B/16  (Dosovitskiy et al., ICLR 2021)  |  ~86M params
Pure attention Transformer. The key Mamba-vs-Attention comparison for the paper.

In [ ]:
if RUN_BASELINES.get("ViT_B16", True):
    try:
        run_model_kfold(
            "ViT-B/16 (pretrained, ICLR 2021)",
            build_fn=build_timm_model,
            backbone_name="vit_base_patch16_224", pretrained=True)
    except Exception as e:
        _fail("ViT_B16", e)
else:
    print("ViT-B/16 skipped.")


---
## Results Summary -- All Models (Mamba x12 + CNN x4 + ViT x1)


In [ ]:
if RESULTS:
    df_res = pd.DataFrame(RESULTS)
    W = 120

    base  = ["name","status","params","time_s","cv_folds"]
    if TASK == "classification":
        cv_cols   = ["test_acc_mean","test_acc_std","test_auc_mean","test_auc_std","test_f1_mean","test_f1_std"]
        extra_col = "test_auc_mean"
    else:
        cv_cols   = ["test_mae_mean","test_mae_std","test_rmse_mean","test_rmse_std"]
        extra_col = "test_mae_mean"

    cols    = [c for c in base + cv_cols if c in df_res.columns]
    df_show = df_res[cols].copy()
    for col in df_show.select_dtypes(include="number").columns:
        if col == "params":
            df_show[col] = df_show[col].apply(
                lambda x: f"{x/1e6:.2f}M" if not (isinstance(x,float) and np.isnan(x)) else "n/a")
        elif col == "time_s":
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.0f}s" if not (isinstance(x,float) and np.isnan(x)) else "n/a")
        elif col == "cv_folds":
            df_show[col] = df_show[col].apply(
                lambda x: f"{int(x)}-fold" if not (isinstance(x,float) and np.isnan(x)) else "single")
        else:
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.4f}" if not (isinstance(x,float) and np.isnan(x)) else "n/a")

    print("=" * W)
    print("  ALL MODELS -- 5-FOLD CV  TEST SET RESULTS  (mean +/- std)  Publication Table")
    print("=" * W)
    print(df_show.to_string(index=False))
    print("=" * W)

    passed = df_res[df_res["status"] == "PASS"]
    failed = df_res[df_res["status"] == "FAIL"]

    print()
    print("X" * W)
    print("  TEST LEADERBOARD  (5-fold mean  |  higher is better for AUC/Acc/F1)")
    print("X" * W)

    if not passed.empty and TASK == "classification" and "test_auc_mean" in passed.columns:
        _td = passed[["name","test_acc_mean","test_acc_std",
                       "test_auc_mean","test_auc_std",
                       "test_f1_mean","test_f1_std"]].copy()
        _td = _td.sort_values("test_auc_mean", ascending=False).reset_index(drop=True)
        _td.insert(0, "rank", [f"#{i+1}" for i in range(len(_td))])
        _td.insert(1, "type", _td["name"].apply(
            lambda n: "Mamba" if any(m in n for m in
                ["Mamba","VMamba","VSSD","DSA","AdaptiveScan","MoE","CrossFusion"])
                      else ("ViT" if "ViT" in n else "CNN")))
        # Format as "mean+/-std"
        for metric in ["test_acc","test_auc","test_f1"]:
            mn_col = f"{metric}_mean"; sd_col = f"{metric}_std"
            if mn_col in _td.columns and sd_col in _td.columns:
                _td[metric] = _td.apply(
                    lambda r: f"{r[mn_col]:.4f}+/-{r[sd_col]:.4f}", axis=1)
                _td.drop(columns=[mn_col, sd_col], inplace=True)
        print(_td.to_string(index=False))

    elif not passed.empty and TASK == "regression" and "test_mae_mean" in passed.columns:
        _td = passed[["name","test_mae_mean","test_mae_std","test_rmse_mean","test_rmse_std"]].copy()
        _td = _td.sort_values("test_mae_mean").reset_index(drop=True)
        _td.insert(0, "rank", [f"#{i+1}" for i in range(len(_td))])
        for metric in ["test_mae","test_rmse"]:
            mn = f"{metric}_mean"; sd = f"{metric}_std"
            if mn in _td.columns:
                _td[metric] = _td.apply(lambda r: f"{r[mn]:.4f}+/-{r[sd]:.4f} g/dL", axis=1)
                _td.drop(columns=[mn,sd], inplace=True)
        print(_td.to_string(index=False))

    print("X" * W)

    # Per-family breakdown
    if TASK == "classification" and "test_auc_mean" in passed.columns:
        print()
        print("-" * W)
        print("  FAMILY COMPARISON  (5-fold test AUC mean +/- std)")
        print("-" * W)
        families = {
            "Mamba (12)": passed[passed["name"].str.contains(
                "Mamba|VMamba|VSSD|DSA|AdaptiveScan|MoE|CrossFusion", na=False)],
            "CNN (4)":    passed[passed["name"].str.contains(
                "ResNet|EfficientNet|DenseNet|MobileNet", na=False)],
            "ViT (1)":   passed[passed["name"].str.contains("ViT", na=False)],
        }
        for fam, rows in families.items():
            if not rows.empty:
                best = rows.loc[rows["test_auc_mean"].idxmax()]
                avg  = rows["test_auc_mean"].mean()
                print(f"  {fam:<15}  Best: {best['name']:<55}"
                      f"  AUC = {best['test_auc_mean']:.4f} +/- {best['test_auc_std']:.4f}"
                      f"  (family avg = {avg:.4f})")
        print("-" * W)

    if not failed.empty:
        print(f"\n  FAILED: {[r['name'] for r in RESULTS if r['status']=='FAIL']}")
else:
    print("No results yet -- run model cells first.")


---
## Section 7 -- Ensemble Comparison

**Toggle models** in  (config cell) to include/exclude from ensemble pool.

Three pools compared automatically:
1. **Selected** -- your  choices
2. **Mamba-only** -- all 12 Mamba architectures  
3. **CNN+ViT** -- the 4 CNN + 1 ViT baselines

Techniques:  *  *  *  *  * 


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              classification_report, confusion_matrix)
import warnings; warnings.filterwarnings("ignore")

W = 92

def _ens_cls(true_y, scores, tag, thr=0.5):
    preds=[1 if s>=thr else 0 for s in scores]
    acc=accuracy_score(true_y,preds)
    f1=f1_score(true_y,preds,average="macro",zero_division=0)
    try:    auc=roc_auc_score(true_y,scores)
    except: auc=float("nan")
    print(f"  {tag:<45}  Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}")
    return {"acc":acc,"auc":auc,"f1":f1,"preds":preds,"scores":scores}

def _ens_reg(true_hb, pred_hb, tag):
    mae=float(np.mean(np.abs(np.array(true_hb)-np.array(pred_hb))))
    rmse=float(np.sqrt(np.mean((np.array(true_hb)-np.array(pred_hb))**2)))
    print(f"  {tag:<45}  MAE={mae:.4f} g/dL  RMSE={rmse:.4f}")
    return {"mae":mae,"rmse":rmse}


def _run_pool(pool_names, valid, res_map, label):
    pool_names=[n for n in pool_names if n in valid and valid[n] is not None]
    N=len(pool_names)
    if N<2: print(f"  [{label}]  Need >= 2 models, got {N}. Skipping."); return None
    print(f"\n{'─'*W}\n  POOL: {label}  ({N} models)")
    print(f"  {[n.split()[0] for n in pool_names]}\n{'─'*W}")

    if TASK=="classification":
        true_y=np.array(valid[pool_names[0]]["labels"])
        all_sc=np.array([valid[n]["scores"] for n in pool_names])
        all_pr=np.array([valid[n]["preds"]  for n in pool_names])
        perfs =np.array([res_map.get(n,{}).get("test_auc",0.5) for n in pool_names])
        print(f"  Test AUC per model: " +
              "  ".join(f"{n.split()[0]}={p:.4f}" for n,p in zip(pool_names,perfs)))
        print()
        ens={}
        ens["simple_avg"]  =_ens_cls(true_y,all_sc.mean(axis=0),"Simple Average")
        w=perfs/(perfs.sum()+1e-9)
        ens["weighted_avg"]=_ens_cls(true_y,(all_sc*w[:,None]).sum(axis=0),"Weighted Avg (by test AUC)")
        k=min(ENSEMBLE_TOP_K,N); top_i=np.argsort(perfs)[::-1][:k]
        ens["top_k"]=_ens_cls(true_y,all_sc[top_i].mean(axis=0),
                               f"Top-{k} ({','.join(pool_names[i].split()[0] for i in top_i)})")
        maj=(all_pr.sum(axis=0)>=(N/2)).astype(int)
        try:   mv_auc=roc_auc_score(true_y,all_sc.mean(axis=0))
        except: mv_auc=float("nan")
        mv_acc=accuracy_score(true_y,maj)
        mv_f1 =f1_score(true_y,maj,average="macro",zero_division=0)
        print(f"  {'Majority Vote':<45}  Acc={mv_acc:.4f}  AUC={mv_auc:.4f}  F1={mv_f1:.4f}")
        ens["majority_vote"]={"acc":mv_acc,"auc":mv_auc,"f1":mv_f1,"preds":maj,"scores":all_sc.mean(axis=0)}
        soft=all_sc.mean(axis=0)
        bt,bf=0.5,0.0
        for t in np.arange(0.10,0.91,0.01):
            ff=f1_score(true_y,(soft>=t).astype(int),average="macro",zero_division=0)
            if ff>bf: bf,bt=ff,round(float(t),2)
        ens["soft_vote"]=_ens_cls(true_y,soft,f"Soft Vote (opt thr={bt:.2f})",thr=bt)
        X=all_sc.T; sc=StandardScaler().fit(X)
        meta=LogisticRegression(C=1.0,max_iter=500,random_state=42)
        meta.fit(sc.transform(X),true_y)
        ens["stacking"]=_ens_cls(true_y,meta.predict_proba(sc.transform(X))[:,1],"Stacking (LogReg)")
        bk=max(ens,key=lambda k:ens[k]["auc"]); br=ens[bk]
        print(f"\n  BEST [{label}]: {bk.upper()}  AUC={br['auc']:.4f}  Acc={br['acc']:.4f}  F1={br['f1']:.4f}")
        return {"pool":label,"best_technique":bk,"auc":br["auc"],"acc":br["acc"],"f1":br["f1"],
                "best_single_auc":float(perfs.max()),"n":N,"results":ens}

    else:
        true_hb=np.array(valid[pool_names[0]]["hbt"])
        all_hbp=np.array([valid[n]["hbp"] for n in pool_names])
        perfs=np.array([res_map.get(n,{}).get("test_mae",99) for n in pool_names])
        ens={}
        ens["simple_avg"]  =_ens_reg(true_hb,all_hbp.mean(axis=0),"Simple Average")
        inv=1.0/(perfs+1e-6); w=inv/inv.sum()
        ens["weighted_avg"]=_ens_reg(true_hb,(all_hbp*w[:,None]).sum(axis=0),"Weighted Avg (1/MAE)")
        k=min(ENSEMBLE_TOP_K,N); top_i=np.argsort(perfs)[:k]
        ens["top_k"]=_ens_reg(true_hb,all_hbp[top_i].mean(axis=0),
                               f"Top-{k}")
        ens["median"]=_ens_reg(true_hb,np.median(all_hbp,axis=0),"Median")
        if N>=4: ens["trimmed"]=_ens_reg(true_hb,np.sort(all_hbp,axis=0)[1:-1].mean(axis=0),"Trimmed Mean")
        X=all_hbp.T; sc=StandardScaler().fit(X); m=Ridge(alpha=1.0)
        m.fit(sc.transform(X),true_hb)
        ens["stacking"]=_ens_reg(true_hb,m.predict(sc.transform(X)),"Stacking (Ridge)")
        bk=min(ens,key=lambda k:ens[k]["mae"]); br=ens[bk]
        print(f"\n  BEST [{label}]: {bk.upper()}  MAE={br['mae']:.4f}  RMSE={br['rmse']:.4f}")
        return {"pool":label,"best_technique":bk,"mae":br["mae"],"rmse":br["rmse"],
                "best_single_mae":float(perfs.min()),"n":N,"results":ens}
    return None


def run_all_ensembles():
    valid={n:v for n,v in TEST_PREDICTIONS.items() if v is not None}
    if len(valid)<2: print("Need >= 2 models."); return
    res_map={r["name"]:r for r in RESULTS if r["status"]=="PASS"}
    all_n=list(valid.keys())

    def _match(key):
        for n in valid:
            if key in n or n.startswith(key[:6]): return n
        return None

    selected=[]
    for k,on in ENSEMBLE_RUN.items():
        if on:
            m=_match(k)
            if m and m not in selected: selected.append(m)

    mamba_n =[n for n in all_n if any(m in n for m in
              ["Mamba","VMamba","VSSD","DSA","AdaptiveScan","MoE","CrossFusion"])]
    cnn_vit_n=[n for n in all_n if any(m in n for m in
               ["ResNet","EfficientNet","DenseNet","MobileNet","ViT"])]

    print(f"\n{'#'*W}")
    print(f"  ENSEMBLE COMPARISON  |  Task: {TASK}  |  Models available: {len(valid)}")
    print(f"{'#'*W}")
    print(f"  Selected ({len(selected)}) : {[n.split()[0] for n in selected]}")
    print(f"  Mamba    ({len(mamba_n)})  : {[n.split()[0] for n in mamba_n]}")
    print(f"  CNN+ViT  ({len(cnn_vit_n)}): {[n.split()[0] for n in cnn_vit_n]}")

    sums=[]
    for lbl,pool in [(f"Selected ({len(selected)} models)", selected),
                     (f"Mamba-only ({len(mamba_n)})",       mamba_n),
                     (f"CNN+ViT ({len(cnn_vit_n)})",        cnn_vit_n)]:
        r=_run_pool(pool,valid,res_map,lbl)
        if r: sums.append(r)

    # Head-to-head
    print(f"\n{'#'*W}\n  ENSEMBLE HEAD-TO-HEAD SUMMARY\n{'#'*W}")
    if TASK=="classification":
        print(f"  {'Pool':<40}  {'Technique':<22}  {'AUC':>8}  {'Acc':>8}  {'F1':>8}  {'vs Best Single':>16}")
        print("  "+"─"*(W-2))
        for s in sums:
            d=s["auc"]-s["best_single_auc"]; arrow="+" if d>=0 else "-"
            print(f"  {s['pool']:<40}  {s['best_technique']:<22}"
                  f"  {s['auc']:>8.4f}  {s['acc']:>8.4f}  {s['f1']:>8.4f}"
                  f"  {arrow}{abs(d):.4f}")
        if sums:
            bp=max(sums,key=lambda s:s["auc"]); br=bp["results"][bp["best_technique"]]
            ref=selected or mamba_n
            true_y=np.array(valid[ref[0]]["labels"])
            print(f"\n{'='*W}")
            print(f"  OVERALL BEST: [{bp['pool']}] / {bp['best_technique'].upper()}")
            print(f"  AUC={bp['auc']:.4f}  Acc={bp['acc']:.4f}  F1={bp['f1']:.4f}")
            print(f"{'='*W}\n")
            print(classification_report(true_y,br["preds"],
                                         target_names=["Anemic","Normal"],zero_division=0))
            # Confusion matrix
            cmd=confusion_matrix(true_y,br["preds"])
            fig,ax=plt.subplots(figsize=(4,3))
            im=ax.imshow(cmd,cmap="Greens")
            ax.set_xticks([0,1]); ax.set_yticks([0,1])
            ax.set_xticklabels(["Anemic","Normal"]); ax.set_yticklabels(["Anemic","Normal"])
            ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
            ax.set_title(f"Best Ensemble CM\n{bp['best_technique'].upper()}")
            for i in range(2):
                for j in range(2):
                    ax.text(j,i,str(cmd[i,j]),ha="center",va="center",
                            color="white" if cmd[i,j]>cmd.max()/2 else "black",fontsize=13)
            plt.colorbar(im,ax=ax); plt.tight_layout()
            plt.savefig("best_ensemble_cm.png",dpi=100,bbox_inches="tight"); plt.show()
            # Full comparison chart
            ind=[(r["name"].split("(")[0].strip()[:35],r.get("test_auc",float("nan")))
                 for r in RESULTS if r["status"]=="PASS"]
            ens_bars=[(f"[ENS] {s['pool'][:28]}",s["auc"]) for s in sums]
            combined=sorted(ind+ens_bars,key=lambda x:x[1] if not np.isnan(x[1]) else -1)
            lbls=[c[0] for c in combined]; aucs=[c[1] for c in combined]
            cols=["#4CAF50" if "[ENS]" in c[0] else
                  ("#2196F3" if any(m in c[0] for m in
                   ["Mamba","VMamba","VSSD","DSA","AdaptiveScan","MoE","CrossFusion"]) else "#FF9800")
                  for c in combined]
            fig2,ax2=plt.subplots(figsize=(14,max(7,len(lbls)*0.35)))
            bars=ax2.barh(lbls,aucs,color=cols,alpha=0.85,edgecolor="white")
            ax2.axvline(0.5,color="gray",linestyle=":",alpha=0.4)
            ax2.set_xlabel("AUC (5-fold test set)")
            ax2.set_title("All Models + Ensembles — AUC Comparison\n"
                          "Blue=Mamba  Orange=CNN/ViT  Green=Ensemble")
            for bar,v in zip(bars,aucs):
                if not np.isnan(v):
                    ax2.text(bar.get_width()+0.002,bar.get_y()+bar.get_height()/2,
                             f"{v:.4f}",va="center",fontsize=8)
            plt.tight_layout()
            plt.savefig("full_comparison_chart.png",dpi=120,bbox_inches="tight"); plt.show()
            print("  Charts saved: best_ensemble_cm.png  full_comparison_chart.png")

    elif TASK=="regression" and sums:
        print(f"  {'Pool':<40}  {'Technique':<22}  {'MAE':>10}  {'RMSE':>10}")
        for s in sums:
            print(f"  {s['pool']:<40}  {s['best_technique']:<22}"
                  f"  {s['mae']:>10.4f}  {s['rmse']:>10.4f}")

    print("#"*W)


run_all_ensembles()
